In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GroupShuffleSplit
import json
import time

print(f"TF version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

CLASS_NAMES = ["DeficienciaNutricional-Boro", "DeficienciaNutricional-Calcio", "DeficienciaNutricional-Fosforo", "DeficienciaNutricional-Hierro", "DeficienciaNutricional-Magnesio", "DeficienciaNutricional-Manganeso", "DeficienciaNutricional-Nitrogeno", "DeficienciaNutricional-Potasio", "Enfermedad-Antracnosis", "Enfermedad-Mancha-de-hierro", "Enfermedad-Roya", "Plaga-AranaRoja", "Plaga-Minador", "Sanas"]
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 100
DATASET = "leaf"
MODEL_NAME = "efficientnetb0"



In [ ]:
# Load images from class folders
from pathlib import Path
import PIL

data_dir = Path("/kaggle/input/coffevision-" + DATASET + "-cls")
if not data_dir.exists():
    # Try alternative paths
    for alt in [Path("/kaggle/working/" + DATASET + "_cls"),
                Path("/kaggle/input/" + DATASET + "_cls")]:
        if alt.exists():
            data_dir = alt
            break

print(f"Data dir: {data_dir}")
print(f"Classes found: {sorted([d.name for d in data_dir.iterdir() if d.is_dir()])}")

# Build file list
records = []
for cls_dir in sorted(data_dir.iterdir()):
    if not cls_dir.is_dir():
        continue
    cls_name = cls_dir.name
    if cls_name not in CLASS_NAMES:
        continue
    for img_path in cls_dir.iterdir():
        if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            records.append({
                'filepath': str(img_path),
                'label': cls_name,
                'group': img_path.stem.split('_')[0],  # group key for split
            })

df = pd.DataFrame(records)
print(f"Total images: {len(df)}")
print(f"Class distribution:\n{df['label'].value_counts()}")

# Map labels to indices
label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
df['label_idx'] = df['label'].map(label_to_idx)

# Split: 70-15-15 using GroupShuffleSplit
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df['group']))
df_train = df.iloc[train_idx]
df_temp = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['group']))
df_val = df_temp.iloc[val_idx]
df_test = df_temp.iloc[test_idx]

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

# Compute class weights
cw = compute_class_weight('balanced', classes=np.array(CLASS_NAMES), y=df_train['label'].values)
class_weight = {i: w for i, w in enumerate(cw)}
print(f"Class weights: {class_weight}")

# Save splits
for name, split_df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    split_df.to_csv(f'/kaggle/working/{DATASET}_{name}.csv', index=False)



In [ ]:
# Build tf.data pipelines
def parse_image(filepath, label_idx):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label_idx

AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices(
    (df_train['filepath'].values, df_train['label_idx'].values)
).map(parse_image, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (df_val['filepath'].values, df_val['label_idx'].values)
).map(parse_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices(
    (df_test['filepath'].values, df_test['label_idx'].values)
).map(parse_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Data augmentation layer
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])



In [ ]:
# Build EfficientNetB0 transfer learning
base_model = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(14, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name="efficientnetb0")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()



In [ ]:
# Train
save_path = f'/kaggle/working/models/{DATASET}_{MODEL_NAME}.keras'
os.makedirs('/kaggle/working/models', exist_ok=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=15, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        save_path, save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.5, patience=5, min_lr=1e-7, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight,
)

# Save final model
model.save(save_path)

# Save training metrics
metrics = {
    'val_accuracy': float(max(history.history['val_accuracy'])),
    'val_loss': float(min(history.history['val_loss'])),
    'best_epoch': int(np.argmax(history.history['val_accuracy'])),
    'total_epochs': len(history.history['accuracy']),
    'final_train_acc': float(history.history['accuracy'][-1]),
    'final_val_acc': float(history.history['val_accuracy'][-1]),
}
with open(f'/kaggle/working/models/{DATASET}_{MODEL_NAME}_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\nTraining complete!")
print(f"Best val_accuracy: {metrics['val_accuracy']:.4f} at epoch {metrics['best_epoch']}")
print(f"Model saved to: {save_path}")



In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Save test metrics
test_metrics = {
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
}
with open(f'/kaggle/working/models/{DATASET}_{MODEL_NAME}_test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)

print("Done! All metrics and models saved to /kaggle/working/models/")

